[//]: # (cr:doc name='project_bootstrap_guided_setup' id=4576d98a)
# Project Bootstrap: Guided Setup

**Purpose:** Configure your retention-analysis project by registering datasets, selecting a prediction objective, and validating temporal feasibility. The output is a single configuration file consumed by all downstream notebooks and pipeline generation.

**What you'll produce:**
- Automatic dataset fingerprinting (entity vs event, target candidates, join keys)
- Prediction objective and anchor selection with evidence
- Merge scaffold for multi-dataset joins
- `project_context.yaml` saved to findings directory

**How to use:** Run cells top-to-bottom. Each section starts with a configuration block — review auto-detected values and override where needed, then run the rest of the cell.

---

## How to Read This Notebook

Each section includes:
- **What it does** — Explanation of the step's purpose
- **What you configure** — Variables at the top of each code cell that you can override
- **What happens automatically** — Logic below that uses your configuration

[//]: # (cr:doc name='0_1_project_metadata' id=cca55d99)
## 0.1 Project Metadata

Give your project a descriptive name. The storage backend is auto-detected from the runtime environment (Databricks vs local). All experiment artifacts are saved under the findings directory.

In [ ]:
# @cr:config name='project_settings' id=846f56cb
# --- Configuration ---
PROJECT_NAME = "email"
LIGHT_RUN = False
MAX_GRID_DATES = None  # e.g. 10 to cap the snapshot grid to 10 dates
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='core_imports' id=a355da3c
# ---------------------

import os
from pathlib import Path

from IPython.display import Markdown, display

from customer_retention.analysis.auto_explorer import initialize_run, mark_notebook
from customer_retention.analysis.notebook_progress import accept_workflow_params

accept_workflow_params()
from customer_retention.analysis.visualization import display_table
from customer_retention.core.compat import native_pd, safe_sample
from customer_retention.core.compat.detection import is_databricks
from customer_retention.core.config.experiments import (
    FINDINGS_DIR,
    get_experiments_dir,
    setup_experiments_structure,
)

_env_grid = os.environ.get("CR_GRID_MAX_DATES")
if _env_grid and MAX_GRID_DATES is None:
    MAX_GRID_DATES = int(_env_grid)

setup_experiments_structure()

_namespace = initialize_run(root=get_experiments_dir(), project_name=PROJECT_NAME)
mark_notebook(_namespace, "00_start_here.ipynb")
RUN_ID = _namespace.run_id
STORAGE_BACKEND = "databricks" if is_databricks() else "local"

display(Markdown(f"""**Project Setup**
- Project: **{PROJECT_NAME}**
- Storage: **{STORAGE_BACKEND}**
- Findings Dir: {FINDINGS_DIR}
"""))


[//]: # (cr:doc name='0_2_dataset_registration' id=f2ccac79)
## 0.2 Dataset Registration

Register all datasets for this project as a dictionary mapping names to file paths or table names. CSV and Parquet files are supported. On Databricks, you can also use Unity Catalog or DLT table names.

In [ ]:
# @cr:config name='dataset_paths' id=f1eb641c
# --- Configuration: dataset names and paths or table names ---
#datasets = {"customer_emails": "/Volumes/churnkit/landing/datasets/customer_emails.csv"}
datasets = {
    "customer_emails": "../tests/fixtures/customer_emails.csv"
}
# datasets = {
#     "customer_retention_retail": "../tests/fixtures/customer_retention_retail.csv"
# }
# datasets = {
#     "customer_profiles": "../tests/fixtures/3set_customer_profiles.csv",
#     "edi_transactions": "../tests/fixtures/3set_edi_transactions.csv",
#     "support_tickets": "../tests/fixtures/3set_support_tickets.csv",
# }
# On Databricks you can also use Unity Catalog / DLT tables:
# datasets = {
#     "customer_profiles": "catalog.schema.customer_profiles",
#     "edi_transactions": "catalog.schema.edi_transactions",
# }
# -------------------------------------------------------------
_namespace.original_datasets = dict(datasets)


In [ ]:
# @cr:code name='resolve_dataset_paths' id=cdc69ccc
from customer_retention.analysis.auto_explorer.dataset_fingerprinter import is_table_name
from customer_retention.core.compat import as_pandas_api, load_spark_table
from customer_retention.core.compat.detection import get_spark_session, is_remote_spark


def _load_source(source: str):
    if is_table_name(source):
        return as_pandas_api(load_spark_table(source))
    p = Path(source)
    if p.exists():
        return native_pd.read_csv(p) if p.suffix == ".csv" else native_pd.read_parquet(p)
    if is_remote_spark():
        spark = get_spark_session()
        if source.endswith(".csv"):
            sdf = spark.read.option("header", "true").option("inferSchema", "true").csv(source)
        else:
            sdf = spark.read.parquet(source)
        return as_pandas_api(sdf)
    return native_pd.read_csv(p) if p.suffix == ".csv" else native_pd.read_parquet(p)


lines = ["**Datasets Registered**"]
for name, source in datasets.items():
    kind = "table" if is_table_name(source) else "file"
    lines.append(f"- **{name}**: {source} ({kind})")
display(Markdown("\n".join(lines)))

In [ ]:
# @cr:code name='replay_registered_landing' id=e2c1f9a3
# Auto-apply registered landing-time transforms (registry.add_landing_filter,
# registry.add_landing_lifecycle_enrichment) onto each dataset. Idempotent:
# datasets that operator Lane-2 cells already mutated to a `global_temp.*`
# view are left alone. With this helper running here, registration alone is
# enough - the operator no longer needs to maintain Lane-2 mutation lines
# in landing-only user_code cells (multi-dataset @cr.register functions
# still run their own Lane-2 because they hold live callables).
from customer_retention.runtime import cr

datasets = cr.replay_registered_landing_steps(
    datasets=datasets,
    registry=registry,
    namespace=_namespace,
    original_datasets=_namespace.original_datasets,
)


In [ ]:
# @cr:code name='snapshot_landing_for_summary' id=5e1a0001
# Capture the post-replay state of `datasets` so the user-code
# assimilation summary at the end of the notebook can detect which
# Lane-2 rebinds happened during this run. Re-runnable.
from customer_retention.runtime import cr

cr.snapshot_landing_state(datasets)

In [ ]:
# @cr:code name='bootstrap_column_descriptions' id=de214303
from customer_retention.stages.causal.column_descriptions_writer import (
    ColumnDescriptionRow,
)
from customer_retention.stages.causal.interpretation import (
    parse_table_descriptions_md,
    write_column_descriptions_sidecar,
)

_cd_candidates = [
    Path.cwd() / "docs" / "sps_table_descriptions.md",
    Path.cwd().parent / "docs" / "sps_table_descriptions.md",
]
_cd_source = next((p for p in _cd_candidates if p.exists()), None)
_cd_rows = []
if _cd_source is not None:
    _cd_rows = parse_table_descriptions_md(_cd_source)

if not _cd_rows:
    # Fallback - ask the Foundation Model to describe each dataset's columns.
    # Works for CSV/Parquet/Delta sources; reuses the same endpoint auto-detect
    # the exploration code uses (`column_describer.resolve_endpoint`).
    try:
        import mlflow.deployments

        from customer_retention.analysis.auto_explorer.column_describer import (
            _build_prompt,
            _parse_llm_response,
            resolve_endpoint,
        )

        _cd_endpoint = resolve_endpoint(None)
        _cd_client = mlflow.deployments.get_deploy_client("databricks")
        # Build the frames dict on demand from `datasets`. The entity-key
        # resolution that produces `_resolved_frames` runs later in NB00,
        # so we resolve sources one at a time here just to read their
        # column lists for the LLM prompt.
        _frames_for_llm = {}
        for _ds_name, _ds_source in datasets.items():
            try:
                _frames_for_llm[_ds_name] = (
                    _ds_source if hasattr(_ds_source, "columns")
                    else _load_source(_ds_source)
                )
            except Exception as _load_exc:
                print(f"Column-descriptions: skipping {_ds_name!r} "
                      f"({type(_load_exc).__name__}: {_load_exc})")
        for _ds_name, _ds_frame in _frames_for_llm.items():
            _cols = list(_ds_frame.columns)
            if not _cols:
                continue
            _label = f"file:{_ds_name}"
            _resp = _cd_client.predict(
                endpoint=_cd_endpoint,
                inputs={
                    "messages": [{"role": "user",
                                  "content": _build_prompt(_label, _cols)}],
                    "max_tokens": 4096, "temperature": 0.1,
                },
            )
            _text = (_resp["choices"][0]["message"]["content"] or "").strip()
            _descs = _parse_llm_response(_text, _cols)
            for _col, _desc in _descs.items():
                _cd_rows.append(ColumnDescriptionRow(
                    table=_ds_name, column_name=_col,
                    business_definition=_desc, source="llm_proposed",
                ))
    except Exception as _cd_exc:
        print(f"Column-descriptions LLM fallback skipped: {type(_cd_exc).__name__}: {_cd_exc}")

if _cd_rows:
    _cd_path = write_column_descriptions_sidecar(_namespace, _cd_rows)
    _source_label = _cd_source.name if _cd_source else "LLM auto-describe"
    display(Markdown(
        f"**Column descriptions registry**: {len(_cd_rows)} rows seeded from "
        f"`{_source_label}` to `{_cd_path}`."
    ))
else:
    display(Markdown(
        "_No markdown seed and LLM fallback produced no rows - column_descriptions "
        "sidecar left empty; the interpretation layer falls back to raw column names._"
    ))

[//]: # (cr:doc name='0_3_auto_fingerprinting' id=b2aaeaa7)
## 0.3 Auto Fingerprinting

Automatically profile each dataset to detect column types, granularity (entity-level vs event-level), entity columns, time columns, and target candidates. Basic statistics like row counts and entity cardinality are computed for each dataset.

Review the summary table — it drives all subsequent auto-detection in this notebook.

In [ ]:
# @cr:code name='fingerprint_datasets' id=43e89806
from customer_retention.analysis.auto_explorer import DatasetFingerprinter

fingerprinter = DatasetFingerprinter(nrows=10000)
fingerprints = fingerprinter.fingerprint_all(datasets)
summary_df = DatasetFingerprinter.to_summary_dataframe(fingerprints)

sampled_names = [fp.name for fp in fingerprints.values() if fp.sampled]
title = "**Dataset Fingerprints**"
if sampled_names:
    title += f"\n\n*Type detection based on first {fingerprinter.nrows:,} rows for: {', '.join(sampled_names)}.*"
display(Markdown(title))
display_table(summary_df)

[//]: # (cr:doc name='0_4_confirm_semantics' id=33cb8ebb)
## 0.4 Confirm Semantics

Review the auto-detected column roles for each dataset:

- **entity_column** — The unique identifier column (e.g., customer_id)
- **time_column** — The primary temporal column (e.g., event_date)
- **raw_time_column_role** — Whether the time column represents event timestamps (for event streams) or update timestamps (for last-modified records)
- **granularity** — Whether the dataset is entity-level (one row per entity) or event-level (multiple rows per entity)
- **target_candidates** — Columns detected as potential churn/target indicators

Override any incorrect detections in the configuration block below.

In [ ]:
# @cr:code name='detect_time_columns' id=ca9650f3
from customer_retention.analysis.auto_explorer.project_context import RawTimeColumnRole

semantics = {}
for name, fp in fingerprints.items():
    role = None
    if fp.time_column and fp.granularity.value == "event_level":
        role = RawTimeColumnRole.EVENT_TIME
    elif fp.time_column:
        role = RawTimeColumnRole.ENTITY_UPDATE_TIME
    semantics[name] = {
        "entity_column": fp.entity_column,
        "time_column": fp.time_column,
        "raw_time_column_role": role,
        "granularity": fp.granularity,
        "target_candidates": fp.target_candidates,
    }

# --- Overrides: uncomment and modify to correct auto-detection ---
# semantics["customer_profiles"]["entity_column"] = "customer_id"
# semantics["edi_transactions"]["raw_time_column_role"] = RawTimeColumnRole.EVENT_TIME
# -----------------------------------------------------------------

_namespace.semantics = semantics

lines = ["**Confirmed Semantics**"]
for name, sem in semantics.items():
    lines.append(f"\n**{name}**")
    for key, val in sem.items():
        display_val = val.value if hasattr(val, "value") else str(val)
        lines.append(f"- {key}: **{display_val}**")
display(Markdown("\n".join(lines)))

[//]: # (cr:doc name='0_5_target_dataset_selection' id=5ad8c8e2)
## 0.5 Target Dataset Selection

Identify which dataset contains the prediction target (e.g., a `churned` column) and which column serves as the entity identifier across all datasets.

The notebook auto-proposes the first dataset with detected target candidates. Override below if the auto-detection is incorrect or if you want to select a different target.

In [ ]:
# @cr:config name='target_overrides' id=1446e49e
# --- Configuration: override auto-detection if needed ---
TARGET_DATASET = None  # e.g., "customer_profiles"
TARGET_COLUMN = None   # e.g., "churned"
ENTITY_COLUMN = None   # e.g., "customer_id"


In [ ]:
# @cr:code name='auto_detect_target' id=637064ae
# Heuristic auto-detection runs ONLY when the operator did not
# explicitly configure TARGET_DATASET / TARGET_COLUMN in the
# `target_overrides` config cell above. If the operator set either,
# the framework treats their configuration as authoritative and
# does NOT attempt to override or replace it. The only auto-fill
# that remains is ENTITY_COLUMN, because every dataset carries one
# and the fingerprint is reliable for it.
_operator_configured = TARGET_DATASET is not None and TARGET_COLUMN is not None

if not _operator_configured:
    # Auto-detection only fires when neither field is set. If the operator
    # set just TARGET_DATASET, don't guess TARGET_COLUMN; if the operator
    # set just TARGET_COLUMN, don't guess TARGET_DATASET.
    if TARGET_DATASET is None and TARGET_COLUMN is None:
        for name, fp in fingerprints.items():
            if fp.target_candidates:
                TARGET_DATASET = name
                TARGET_COLUMN = fp.target_candidates[0]
                ENTITY_COLUMN = ENTITY_COLUMN or fp.entity_column
                break

if ENTITY_COLUMN is None:
    if TARGET_DATASET and TARGET_DATASET in fingerprints:
        _fp = fingerprints[TARGET_DATASET]
        if _fp.entity_column:
            ENTITY_COLUMN = _fp.entity_column
    if ENTITY_COLUMN is None:
        for fp in fingerprints.values():
            if fp.entity_column:
                ENTITY_COLUMN = fp.entity_column
                break

lines = ["**Target Selection**"]
lines.append(f"- Source: **{'operator config' if _operator_configured else 'auto-detect'}**")
lines.append(f"- Target Dataset: **{TARGET_DATASET or 'NOT SET'}**")
lines.append(f"- Target Column: **{TARGET_COLUMN or 'NOT SET'}**")
lines.append(f"- Entity Column: **{ENTITY_COLUMN or 'NOT SET'}**")
if not TARGET_DATASET:
    lines.append("\n> **Warning:** No target dataset detected. Set TARGET_DATASET / TARGET_COLUMN above.")
display(Markdown("\n".join(lines)))

[//]: # (cr:doc name='0_6_prediction_objective_detection' id=1feda677)
## 0.6 Prediction Objective Detection

Assess which prediction objectives your data can support. Each objective is scored by confidence and ranked automatically. All objectives are tracked — you can adjust priorities in the next section.

In [ ]:
# @cr:code name='detect_prediction_objective' id=f13e88da
from customer_retention.analysis.auto_explorer import PredictionObjectiveDetector
from customer_retention.analysis.auto_explorer.project_context import (
    ObjectiveAssessment,
    ObjectivePriority,
    ObjectiveSpec,
)

detector = PredictionObjectiveDetector()

target_fp = fingerprints.get(TARGET_DATASET)
if target_fp is not None:
    target_data = datasets[TARGET_DATASET]
    if isinstance(target_data, str):
        target_df = _load_source(target_data)
    else:
        target_df = target_data

    time_col_for_detect = target_fp.time_column
    raw_assessments = detector.detect_feasible_objectives(
        target_df, ENTITY_COLUMN or target_fp.entity_column, TARGET_COLUMN, time_col_for_detect,
    )
else:
    raw_assessments = []

feasible_sorted = sorted(
    [a for a in raw_assessments if a.feasible],
    key=lambda a: a.confidence,
    reverse=True,
)
infeasible = [a for a in raw_assessments if not a.feasible]

_priority_order = [ObjectivePriority.PRIMARY, ObjectivePriority.SECONDARY, ObjectivePriority.EXPLORATORY]
objective_specs = []
for idx, a in enumerate(feasible_sorted):
    priority = _priority_order[min(idx, len(_priority_order) - 1)]
    objective_specs.append(ObjectiveSpec(
        objective=a.objective,
        priority=priority,
        anchor=a.suggested_anchor,
        parameters=a.parameters,
        assessment=ObjectiveAssessment(
            confidence=round(a.confidence * 100),
            suggested_anchor=a.suggested_anchor,
            rationale=a.evidence,
            feasibility=a.parameters or None,
        ),
    ))

rows = []
for spec in objective_specs:
    rows.append({
        "objective": spec.objective.value,
        "priority": spec.priority.value,
        "confidence": f"{spec.assessment.confidence}%",
        "anchor": spec.effective_anchor.value if spec.effective_anchor else "-",
        "key_evidence": spec.assessment.rationale[0] if spec.assessment.rationale else "-",
    })
for a in infeasible:
    rows.append({
        "objective": a.objective.value,
        "priority": "disabled",
        "confidence": f"{a.confidence:.0%}",
        "anchor": a.suggested_anchor.value,
        "key_evidence": a.evidence[0] if a.evidence else "-",
    })

display(Markdown("**Prediction Objective Analysis**"))
if rows:
    display_table(native_pd.DataFrame(rows))
else:
    display(Markdown("> **Warning:** No objectives detected. Review your data."))

[//]: # (cr:doc name='0_7_objective_priority_review' id=aadb9261)
## 0.7 Objective Priority Review

Review the auto-assigned priorities and override if needed. All feasible objectives are tracked throughout exploration — the primary objective drives downstream notebooks, while secondary and exploratory objectives continue collecting evidence.

Priority levels:
  - **PRIMARY** — main focus for label building, cohort definitions, and training
  - **SECONDARY** — actively explored, ready for activation later
  - **EXPLORATORY** — evidence collected but not yet validated
  - **DISABLED** — excluded from analysis

In [ ]:
# @cr:code name='configure_prediction_anchor' id=85a730b7
from customer_retention.analysis.auto_explorer.project_context import PredictionAnchor, PredictionObjective

# --- Configuration: override priorities or anchors ---
# To change priority: find the spec by objective and reassign
# objective_specs[0].priority = ObjectivePriority.SECONDARY
# To swap primary: set old primary to SECONDARY, new to PRIMARY
# To disable: spec.priority = ObjectivePriority.DISABLED
# To override anchor: spec.anchor = PredictionAnchor.CONTRACT
# To add business parameters:
# spec.parameters["prediction_horizon_days"] = 90
# -----------------------------------------------------

primary_specs = [s for s in objective_specs if s.priority == ObjectivePriority.PRIMARY]
PRIMARY_OBJECTIVE = primary_specs[0].objective if primary_specs else PredictionObjective.IMMEDIATE_RISK

lines = ["**Objective Priorities**"]
for spec in objective_specs:
    marker = " <-- primary" if spec.priority == ObjectivePriority.PRIMARY else ""
    anchor_display = spec.effective_anchor.value if spec.effective_anchor else "unset"
    lines.append(f"- **{spec.objective.value}**: {spec.priority.value} (anchor: {anchor_display}){marker}")
    if spec.parameters:
        for k, v in spec.parameters.items():
            lines.append(f"  - {k}: **{v}**")
display(Markdown("\n".join(lines)))

[//]: # (cr:doc name='0_8_join_scaffold' id=b7b31d8b)
## 0.8 Join Scaffold

Detect relationships between datasets by comparing column names and value overlaps. For each non-target dataset, the best join key and relationship type (one-to-one, one-to-many, etc.) are identified against the target dataset.

The resulting merge scaffold defines how datasets will be joined during feature engineering. Review the detected joins — you can manually exclude datasets after this cell runs.

In [ ]:
# @cr:config name='merge_scaffold_overrides' id=2a900807
from customer_retention.analysis.auto_explorer.project_context import MergeScaffoldEntry

# --- Configuration: override or replace auto-detected joins ---
# Set to a list of MergeScaffoldEntry to skip auto-detection entirely:
# MANUAL_SCAFFOLD = [
#     MergeScaffoldEntry(left_dataset="edi_transactions", right_dataset="customer_profiles",
#                        join_keys=["customer_id"], relationship="many_to_one"),
#     MergeScaffoldEntry(left_dataset="support_tickets", right_dataset="customer_profiles",
#                        join_keys=["customer_id"], relationship="many_to_one"),
# ]
MANUAL_SCAFFOLD = None
# To exclude datasets after auto-detection, add names here:
EXCLUDE_DATASETS = []  # e.g., ["support_tickets"]
# --------------------------------------------------------------

In [ ]:
# @cr:code name='detect_relationships' id=7f4b562b
from customer_retention.core.compat import is_dataframe
from customer_retention.stages.profiling.relationship_detector import RelationshipDetector

loaded_frames = {}
for name, source in datasets.items():
    if is_dataframe(source):
        loaded_frames[name] = source
    else:
        loaded_frames[name] = _load_source(source)

if MANUAL_SCAFFOLD is not None:
    merge_scaffold = MANUAL_SCAFFOLD
else:
    rel_detector = RelationshipDetector()
    merge_scaffold = []
    if TARGET_DATASET and TARGET_DATASET in loaded_frames:
        target_frame = loaded_frames[TARGET_DATASET]
        for name, frame in loaded_frames.items():
            if name == TARGET_DATASET or name in EXCLUDE_DATASETS:
                continue
            rel = rel_detector.detect(frame, target_frame, df1_name=name, df2_name=TARGET_DATASET)
            if rel.suggested_join:
                merge_scaffold.append(MergeScaffoldEntry(
                    left_dataset=name,
                    right_dataset=TARGET_DATASET,
                    join_keys=[rel.suggested_join.left_column],
                    relationship=rel.relationship_type.value,
                ))

lines = ["**Join Scaffold**"]
if merge_scaffold:
    for entry in merge_scaffold:
        keys_str = ", ".join(f"`{k}`" for k in entry.join_keys)
        lines.append(f"- **{entry.left_dataset}** -> **{entry.right_dataset}** on {keys_str} ({entry.relationship})")
else:
    lines.append("- No join relationships detected (single dataset or no shared keys)")
display(Markdown("\n".join(lines)))

[//]: # (cr:doc name='0_8_1_key_resolution_bridge_datasets' id=a7f95e46)
## 0.8.1 Key Resolution (Bridge Datasets)

Some datasets do not contain the entity column directly but can reach it through
a bridge table (e.g., `case_history` reaches `ACCOUNT_ID` via the `case` table's
`CASE_ID` → `ACCOUNT_ID` mapping).

Auto-detection scans for `_ID` / `_KEY` columns shared between datasets and picks
the bridge with the best key overlap. Override with `KEY_RESOLUTION` to define
explicit paths or skip auto-detection.

In [ ]:
# @cr:config name='key_resolution_overrides' id=7b1ec50d
from customer_retention.analysis.auto_explorer.project_context import KeyResolutionStep

# --- Configuration: manual key resolution override ---
# KEY_RESOLUTION = {
#     "case_history": [
#         KeyResolutionStep(bridge_dataset="case", source_key="CASE_ID",
#                           bridge_key="CASE_ID", resolve_column="ACCOUNT_ID"),
#     ],
# }
KEY_RESOLUTION = None

In [ ]:
# @cr:code name='apply_key_resolutions' id=0866aa83
from customer_retention.analysis.auto_explorer import suggest_key_resolutions

if KEY_RESOLUTION is not None:
    key_resolutions = KEY_RESOLUTION
elif ENTITY_COLUMN:
    key_resolutions = suggest_key_resolutions(loaded_frames, ENTITY_COLUMN)
else:
    key_resolutions = {}

lines = ["**Key Resolution (Bridge Datasets)**"]
if key_resolutions:
    for ds_name, steps in key_resolutions.items():
        for step in steps:
            lines.append(
                f"- **{ds_name}**: `{step.source_key}` → "
                f"**{step.bridge_dataset}**.`{step.bridge_key}` → `{step.resolve_column}`"
            )
else:
    lines.append("- All datasets already contain the entity column — no bridges needed")
display(Markdown("\n".join(lines)))

[//]: # (cr:doc name='0_8_2_dataset_registry' id=137df52e)
## 0.8.2 Dataset Registry

Build registry entries for each dataset, combining fingerprints, semantics, scaffold
join info, and key resolution. This registry is the data-structure half of the project
context — temporal parameters (posture, intent) are added in later sections.

In [ ]:
# @cr:code name='build_dataset_registry' id=915bcef8
from customer_retention.analysis.auto_explorer.project_context import DatasetRegistryEntry
from customer_retention.core.config.column_config import DatasetGranularity


def _detect_storage_format(source: str) -> str:
    if is_table_name(source):
        return "delta"
    p = Path(source)
    return "parquet" if p.suffix == ".parquet" else "csv"


# Exploration-side registry. `datasets[name]` here is whatever the
# preceding registered-landing replay (and any operator Lane-2 cells)
# produced - typically a `global_temp.*` view name when a filter or
# lifecycle enrichment was registered for the dataset, otherwise the
# upstream UC table. NB01a reads `_ds_entry.path` from this registry so
# pointing it at the post-mutation view is what lets fingerprinting see
# the enriched columns (`event_timestamp`, `event_type`, etc.).
#
# Production lineage lives separately on `_namespace.original_datasets`
# (stashed by cell `f1eb641c`) and is propagated to NB10/codegen via
# `ProjectContext.original_datasets`. The guard below fails fast when a
# `global_temp.*` value is in `datasets[name]` AND that name is missing
# from `_namespace.original_datasets`: NB10 would have nothing to replay
# landing from, and the generated landing would crash on
# `spark.read.table("global_temp.*")`.
_originals = dict(getattr(_namespace, "original_datasets", None) or {})
_session_local_leaks = sorted(
    name for name, src in datasets.items()
    if isinstance(src, str) and src.startswith("global_temp.") and name not in _originals
)
if _session_local_leaks:
    raise RuntimeError(
        "build_dataset_registry: NB10 codegen will not be able to replay "
        f"landing for {_session_local_leaks!r} - their `datasets[name]` "
        "is a session-scoped `global_temp.*` view AND `_namespace."
        "original_datasets` has no upstream path to read from. Re-run "
        "cell `f1eb641c` (dataset_paths) so it stashes `_namespace."
        "original_datasets = dict(datasets)` BEFORE any user_code "
        "mutation, or remove the upstream `register_temp_view` call and "
        "instead register the transform via `registry.add_landing_filter("
        "dataset=..., predicate=...)` / `registry.add_landing_lifecycle_"
        "enrichment(dataset=..., config=...)` so codegen can replay it."
    )


registry = {}
for name, fp in fingerprints.items():
    sem = semantics[name]
    is_target = name == TARGET_DATASET

    join_keys, join_to, relationship = [], None, None
    for ms in merge_scaffold:
        if ms.left_dataset == name:
            join_keys = list(ms.join_keys)
            join_to = ms.right_dataset
            relationship = ms.relationship
            break

    source = datasets[name]
    registry[name] = DatasetRegistryEntry(
        name=name,
        path=str(source) if isinstance(source, str) else name,
        storage_format=_detect_storage_format(str(source)) if isinstance(source, str) else "dataframe",
        entity_column=sem["entity_column"],
        time_column=sem["time_column"],
        raw_time_column_role=sem["raw_time_column_role"],
        granularity=sem["granularity"],
        row_count=fp.row_count,
        unique_entities=fp.unique_entities,
        avg_rows_per_entity=fp.avg_rows_per_entity,
        target_candidates=fp.target_candidates,
        role="target" if is_target else "feature_source" if join_keys else None,
        join_keys=join_keys,
        join_to=join_to,
        relationship=relationship,
    )

for ds_name, steps in key_resolutions.items():
    if ds_name in registry:
        registry[ds_name].key_resolution = list(steps)

_bridge_count = sum(1 for r in registry.values() if r.key_resolution)

# Resolve entity keys in-memory and save resolved datasets to landing.
# All frames are already loaded - no need to persist bridge datasets separately.
if key_resolutions:
    from customer_retention.analysis.auto_explorer.active_dataset_store import save_active_dataset
    from customer_retention.analysis.auto_explorer.key_resolver import resolve_entity_keys
    _resolved_frames = resolve_entity_keys(loaded_frames, key_resolutions)
    for _ds_name in sorted(key_resolutions):
        save_active_dataset(_namespace, _ds_name, _resolved_frames[_ds_name])

_namespace.dataset_registry = registry

lines = ["**Dataset Registry**"]
lines.append(f"- Datasets: **{len(registry)}**")
lines.append(f"- Bridge resolutions: **{_bridge_count}**")
if key_resolutions:
    lines.append(f"- Resolved and saved to landing: **{', '.join(sorted(key_resolutions))}**")
for name, entry in registry.items():
    role = entry.role or "unlinked"
    bridge = f" (via {entry.key_resolution[0].bridge_dataset})" if entry.key_resolution else ""
    lines.append(f"- **{name}**: {entry.granularity.value}, {role}{bridge}")
display(Markdown("\n".join(lines)))


[//]: # (cr:doc name='0_9_temporal_posture' id=6af52c1f)
## 0.9 Temporal Posture

Select how much temporal history the model considers:

- **LONG_MEMORY** — Use extended historical context for stable, long-range patterns. Best for initial projects or when data changes slowly.
- **SHORT_MEMORY** — Focus on recent data for fast-adapting, short-range patterns. Best when data distribution shifts over time.

In [ ]:
# @cr:config name='temporal_posture_config' id=09a0854f
from customer_retention.analysis.auto_explorer.project_context import TemporalPosture

# --- Configuration: select temporal posture ---
TEMPORAL_POSTURE = TemporalPosture.STABLE


In [ ]:
# @cr:code name='display_temporal_posture' id=04cf48a9
# ----------------------------------------------

display(Markdown(f"**Temporal Posture:** **{TEMPORAL_POSTURE.value}**"))

[//]: # (cr:doc name='0_10_intent_configuration' id=b3f8f766)
## 0.10 Intent Configuration

Configure the prediction intent for this run. The intent captures temporal parameters that flow through all downstream notebooks:

| Parameter | Description |
|-----------|-------------|
| **Prediction Horizons** | List of horizons (days) to evaluate; primary horizon is the largest |
| **Recent Window** | How many days of history to use for features |
| **Observation Window** | Lookback for feature aggregation (equals recent window) |
| **Purge Gap** | Days between feature cutoff and label start (prevents leakage) |
| **Label Window** | Days after purge gap in which label is observed |
| **Temporal Split** | Whether to use time-based train/test splitting |
| **Cadence Interval** | Retraining / scoring frequency (daily, weekly, biweekly, monthly) |
| **Split Strategy** | Train/test splitting approach (temporal or cohort-based) |

Defaults are computed from the objective, posture, and prediction horizon using the `IntentDefaultsEngine`. Override any value in the configuration block below.

In [ ]:
# @cr:config name='intent_config' id=ea85abf2
from customer_retention.analysis.auto_explorer import CadenceInterval, IntentConfig, IntentDefaultsEngine, SplitStrategy

engine = IntentDefaultsEngine()
_data_span = max((fp.temporal_span_days or 0) for fp in fingerprints.values()) or None

# --- Configuration: prediction horizon (days) ---
PREDICTION_HORIZON = 90
# -------------------------------------------------

suggested = engine.suggest(
    objective=PRIMARY_OBJECTIVE,
    posture=TEMPORAL_POSTURE,
    prediction_horizon=PREDICTION_HORIZON,
    data_span_days=_data_span,
)

# --- Configuration: override suggested defaults ---
PREDICTION_HORIZONS = suggested.config.prediction_horizons
RECENT_WINDOW_DAYS = suggested.config.recent_window_days
OBSERVATION_WINDOW_DAYS = suggested.config.observation_window_days
PURGE_GAP_DAYS = suggested.config.purge_gap_days
LABEL_WINDOW_DAYS = suggested.config.label_window_days
TEMPORAL_SPLIT = suggested.config.temporal_split
CADENCE_INTERVAL = suggested.config.cadence_interval
SPLIT_STRATEGY = suggested.config.split_strategy
HISTORY_UPPER_LIMIT = None #   HISTORY_UPPER_LIMIT = "2024-12-31" means ignore data after 2024, None means use all data up to the latest date
LOOKBACK_PERIODS = None # None means use all data, otherwise keep only the specified number of periods before the upper limit


In [ ]:
# @cr:code name='build_intent' id=a7f7bf52
# ---------------------------------------------------

intent = IntentConfig(
    prediction_horizons=PREDICTION_HORIZONS,
    recent_window_days=RECENT_WINDOW_DAYS,
    observation_window_days=OBSERVATION_WINDOW_DAYS,
    purge_gap_days=PURGE_GAP_DAYS,
    label_window_days=LABEL_WINDOW_DAYS,
    temporal_split=TEMPORAL_SPLIT,
    cadence_interval=CADENCE_INTERVAL,
    split_strategy=SPLIT_STRATEGY,
    history_upper_limit=HISTORY_UPPER_LIMIT,
    lookback_periods=LOOKBACK_PERIODS,
)

rows = [
    {"Parameter": "Prediction Horizons", "Value": str(intent.prediction_horizons),
     "Formula": suggested.formula_explanations["prediction_horizons"]},
    {"Parameter": "Recent Window (days)", "Value": intent.recent_window_days,
     "Formula": suggested.formula_explanations["recent_window_days"]},
    {"Parameter": "Observation Window (days)", "Value": intent.observation_window_days,
     "Formula": suggested.formula_explanations["observation_window_days"]},
    {"Parameter": "Purge Gap (days)", "Value": intent.purge_gap_days,
     "Formula": suggested.formula_explanations["purge_gap_days"]},
    {"Parameter": "Label Window (days)", "Value": intent.label_window_days,
     "Formula": suggested.formula_explanations["label_window_days"]},
    {"Parameter": "Cadence Interval", "Value": intent.cadence_interval.value,
     "Formula": suggested.formula_explanations["cadence_interval"]},
    {"Parameter": "Split Strategy", "Value": intent.split_strategy.value,
     "Formula": suggested.formula_explanations["split_strategy"]},
    {"Parameter": "History Upper Limit", "Value": intent.history_upper_limit or "latest",
     "Formula": "user-specified or latest date in data"},
    {"Parameter": "Lookback Periods", "Value": str(intent.lookback_periods) if intent.lookback_periods else "all data",
     "Formula": f"in {intent.cadence_interval.value} units"},
]
display(Markdown("**Intent Configuration**"))
display_table(native_pd.DataFrame(rows))

from customer_retention.analysis.auto_explorer.intent_defaults import format_dataset_preview

_target_fp = fingerprints.get(TARGET_DATASET) if TARGET_DATASET else None
if _target_fp and _target_fp.temporal_span_days and _target_fp.unique_entities:
    display(Markdown(format_dataset_preview(
        _target_fp.temporal_span_days, intent.cadence_interval,
        _target_fp.unique_entities, _target_fp.avg_rows_per_entity or 0,
    )))

[//]: # (cr:doc name='0_11_initialize_snapshot_grid' id=081e4d57)
## 0.11 Initialize Snapshot Grid

Create the snapshot grid that defines the temporal grid for entity x as_of_date snapshots. The grid is used by notebook 1d to produce time-aware entity-level aggregations.

**Grid Modes:**
- **NO_ADJUSTMENTS** (default): Grid parameters are fixed from the intent config. Any dataset reaching 1d can be aggregated immediately.
- **ALLOW_ADJUSTMENTS**: Grid can be modified by votes from notebooks 01a-01c. Step 1d blocks until all event datasets have voted.


In [ ]:
# @cr:config name='snapshot_grid_config' id=2ef9c4c6
from customer_retention.analysis.auto_explorer.snapshot_grid import GridAdjustmentMode, SnapshotGrid

# --- Configuration: grid mode ---
GRID_MODE = GridAdjustmentMode.NO_ADJUSTMENTS


In [ ]:
# @cr:code name='build_snapshot_grid' id=6b061a37
# ---------------------------------

snapshot_grid = SnapshotGrid.from_intent(
    intent=intent,
    granularities={name: sem["granularity"] for name, sem in semantics.items()},
    mode=GRID_MODE,
    fingerprints=fingerprints,
)

if MAX_GRID_DATES is not None:
    snapshot_grid.max_grid_dates = MAX_GRID_DATES
    os.environ["CR_GRID_MAX_DATES"] = str(MAX_GRID_DATES)

snapshot_grid.save(_namespace.snapshot_grid_path)

_namespace.snapshot_grid = snapshot_grid

_event_votes = [n for n, v in snapshot_grid.dataset_votes.items() if not v.voted]
_entity_votes = [n for n, v in snapshot_grid.dataset_votes.items() if v.voted]
_boundary_label = (
    f"**{snapshot_grid.grid_start}** to **{snapshot_grid.grid_end}**"
    if snapshot_grid.grid_start and snapshot_grid.grid_end
    else "not yet computed (will be set in 01d from dataset votes)"
)
_grid_cap = f" (capped to {MAX_GRID_DATES})" if MAX_GRID_DATES else ""
_hw_label = "full data"
if intent.history_upper_limit or intent.lookback_periods:
    _parts = []
    if intent.lookback_periods:
        _parts.append(f"last {intent.lookback_periods} {intent.cadence_interval.value} periods")
    if intent.history_upper_limit:
        _parts.append(f"up to {intent.history_upper_limit}")
    _hw_label = ", ".join(_parts)
display(Markdown(f"""**Snapshot Grid Initialized**
- Mode: **{snapshot_grid.mode.value}**
- Cadence: **{snapshot_grid.cadence_interval.value}** ({snapshot_grid.cadence_to_days()} days)
- Observation Window: **{snapshot_grid.observation_window_days}** days
- Grid Boundaries: {_boundary_label}
- History Window: **{_hw_label}**
- Max Grid Dates: **{MAX_GRID_DATES or 'unlimited'}**{_grid_cap}
- Datasets auto-voted (entity-level): {_entity_votes or 'none'}
- Datasets awaiting vote (event-level): {_event_votes or 'none'}
- Saved to: {_namespace.snapshot_grid_path}
"""))

[//]: # (cr:doc name='0_12_save_project_context' id=4874dfcd)
## 0.12 Save Project Context

Assemble all configuration into a single context file and save it. This file is the single source of truth for all downstream notebooks (data discovery, feature engineering, training, scoring) and pipeline generation.

The context includes an exploration contract that enforces two governance rules:
1. **Dual view** — Every exploration notebook must produce both a full-history view and a recent-behavior view
2. **Insight mapping** — Every analysis insight must be tagged with which prediction objective it serves

In [ ]:
# @cr:code name='build_exploration_contract' id=c6211778

from customer_retention.analysis.auto_explorer.project_context import (
    ExplorationContract,
    ProjectContext,
)

project_context = ProjectContext(
    project_name=PROJECT_NAME,
    run_id=RUN_ID,
    storage_backend=STORAGE_BACKEND,
    datasets=registry,
    original_datasets=dict(_namespace.original_datasets or {}),
    target_dataset=TARGET_DATASET,
    target_column=TARGET_COLUMN,
    entity_column=ENTITY_COLUMN,
    objectives=objective_specs,
    primary_objective=PRIMARY_OBJECTIVE,
    temporal_posture=TEMPORAL_POSTURE,
    merge_scaffold=merge_scaffold,
    exploration_contract=ExplorationContract(),
    intent=intent,
    light_run=LIGHT_RUN,
)

context_path = _namespace.project_context_path
project_context.save(context_path)

from customer_retention.analysis.notebook_progress import publish_workflow_metadata

publish_workflow_metadata(project_context)

target_label = f"**{TARGET_DATASET}**.{TARGET_COLUMN}" if TARGET_DATASET else "NOT SET"
active = project_context.active_objectives
obj_summary = ", ".join(f"{o.objective.value} ({o.priority.value})" for o in active)
_light_label = "ON" if LIGHT_RUN else "OFF"
_grid_label = f"{MAX_GRID_DATES} dates" if MAX_GRID_DATES else "OFF (full grid)"
display(Markdown(f"""**Project Context Saved**
- Path: {context_path}
- Datasets: **{len(project_context.datasets)}**
- Target: {target_label}
- Primary Objective: **{PRIMARY_OBJECTIVE.value}**
- All Objectives: {obj_summary}
- Posture: **{TEMPORAL_POSTURE.value}**
- Intent: recent={intent.recent_window_days}d, purge={intent.purge_gap_days}d, label={intent.label_window_days}d, cadence={intent.cadence_interval.value}, split={intent.split_strategy.value}
- Light Run: **{_light_label}**
- Max Grid Dates: **{_grid_label}**
- Contract: dual-view + insight mapping
"""))


[//]: # (cr:doc name='0_13_exploration_sampling' id=d3a1f2e0)
## 0.13 Exploration Sampling

Choose how many entities to sample for exploration. Sampling preserves target
class balance, temporal cohort coverage, and optionally user-specified column
distributions. All downstream notebooks operate on this sample. Production
pipelines always use full data.


In [ ]:
# @cr:config name='sampling_config' id=e4b7c9a1
SAMPLE_ENTITY_COUNT = None          # set after reviewing table below (e.g. 5000)
HOLDOUT_FRACTION = 0.1              # fraction of sampled entities reserved for holdout validation.
                                    # IDs are persisted to runs/<run_id>/holdout_entity_ids.json — that file is
                                    # THE source of truth for the holdout. The FeatureSpec written by NB08
                                    # (merged/feature_spec.yaml) only *references* the path; it does not
                                    # duplicate the IDs. Keep the file intact; both NB08 training and the
                                    # generated production pipeline read from it via RunNamespace.
SAMPLE_STRATIFY_COLUMNS = []        # extra columns to stratify by (e.g. ["region"])
SAMPLE_FILTER_COLUMNS = {}          # per-dataset segment filters using query syntax
                                    # entities with ANY non-matching row are excluded entirely, e.g.
                                    # {"customers": "region in ['US', 'UK']",
                                    #  "transactions": "amount > 0 and status != 'cancelled'"}

_env_sample = os.environ.get("CR_SAMPLE_ENTITY_COUNT")
if _env_sample and SAMPLE_ENTITY_COUNT is None:
    SAMPLE_ENTITY_COUNT = int(_env_sample)


In [ ]:
# @cr:code name='validate_sample_filter_funnel' id=c19f8739
# Per-predicate breakdown of SAMPLE_FILTER_COLUMNS against raw loaded frames —
# visible funnel of total → each filter → combined, before sampling runs.
# Auto-decomposes the account filter into its canonical SPS pieces (segment +
# contract-start subquery). Adapt the `predicates` list if your filter differs.
if SAMPLE_FILTER_COLUMNS and TARGET_DATASET in loaded_frames:
    import pyspark.sql.functions as F  # noqa: N812

    from customer_retention.core.compat import as_spark_df

    _raw_account = as_spark_df(loaded_frames[TARGET_DATASET])
    _total = _raw_account.select(ENTITY_COLUMN).distinct().count()
    print(f"[filter-funnel] total {TARGET_DATASET} entities: {_total:,}")

    predicates = []
    if "REVENUE_MARKET_SEGMENT" in _raw_account.columns:
        predicates.append((
            "segment", "REVENUE_MARKET_SEGMENT in ('Emerging', 'Small')",
            _raw_account.filter(F.col("REVENUE_MARKET_SEGMENT").isin("Emerging", "Small"))
                         .select(ENTITY_COLUMN).distinct(),
        ))
    _raw_contract = loaded_frames.get("contract")
    if _raw_contract is not None:
        _rc = as_spark_df(_raw_contract)
        if "event_type" in _rc.columns:
            predicates.append((
                "contract_start", "ACCOUNT_ID in (select ACCOUNT_ID from contract where event_type='start')",
                _rc.filter(F.col("event_type") == "start").select("ACCOUNT_ID").distinct(),
            ))

    _combined_ids = None
    for label, expr, ids_df in predicates:
        n = ids_df.count()
        pct = (n / _total * 100) if _total else 0.0
        print(f"  [{label}] {expr}  →  {n:,}  ({pct:.1f}% of total)")
        _combined_ids = ids_df if _combined_ids is None else _combined_ids.join(ids_df, ENTITY_COLUMN, "inner")

    if _combined_ids is not None:
        _combined_n = _combined_ids.count()
        _pct = (_combined_n / _total * 100) if _total else 0.0
        print(f"  [COMBINED] both predicates pass  →  {_combined_n:,}  ({_pct:.2f}% of total)")
        if _combined_n < max(100, int(_total * 0.01)):
            print("  \u26a0\ufe0f  combined filter passes < 1% of total — confirm this is intended")
else:
    print("[filter-funnel] skipped: SAMPLE_FILTER_COLUMNS empty or TARGET_DATASET not loaded")


In [ ]:
# @cr:code name='run_sampling' id=f5c8d6b2
from customer_retention.analysis.auto_explorer.sampling import (
    candidate_sample_sizes,
    compute_sampling_universe,
    estimate_sampling_accuracy,
    prepare_sample_frame,
    render_population_markdown,
    render_sample_result_markdown,
    render_segment_filter_markdown,
    save_sample_ids,
    stratified_entity_sample,
    stratified_holdout_split,
    summarize_population,
)

_primary = TARGET_DATASET if TARGET_DATASET and TARGET_DATASET in loaded_frames \
    else next(iter(loaded_frames), None)
if _primary is None:
    raise ValueError("loaded_frames is empty — register at least one dataset before sampling")
_target_name = _primary
_target_df = loaded_frames.get(_target_name)
_time_col = semantics.get(_target_name, {}).get("time_column") if _target_name else None
_population = summarize_population(_target_df, ENTITY_COLUMN, TARGET_COLUMN, _time_col)

_segment_entity_cols = {name: reg.entity_column for name, reg in registry.items() if reg.entity_column}
if _primary not in _segment_entity_cols:
    raise ValueError(
        f"primary dataset {_primary!r} has no entity_column in registry — "
        f"cannot anchor sampling universe"
    )

# C1b — ANCHOR principle applied to temporal lookback.
# NB01 re-applies apply_temporal_lookback(primary, ...) at data-load time,
# which otherwise drops ~58% of sample IDs silently (same orphan shape
# C1 fixed, different narrowing layer). Applying lookback here first
# ensures the sampling universe is already lookback-passing — every
# sample ID then survives NB01's re-application.
_primary_time_col = semantics.get(_primary, {}).get("time_column")
_primary_role = semantics.get(_primary, {}).get("raw_time_column_role")
_should_apply_lookback = (
    _primary_time_col is not None
    and _primary_time_col in loaded_frames[_primary].columns
    and getattr(intent, "lookback_periods", None)
    and (_primary_role is None or getattr(_primary_role, "should_apply_lookback", True))
)
if _should_apply_lookback:
    from customer_retention.analysis.auto_explorer.sampling import (
        apply_temporal_lookback,
    )
    _pre_lookback_count = len(loaded_frames[_primary])
    loaded_frames[_primary] = apply_temporal_lookback(
        loaded_frames[_primary], _primary_time_col, intent,
    )
    _post_lookback_count = len(loaded_frames[_primary])
    print(
        f"[C1b] pre-sampling lookback on {_primary!r}: "
        f"{_pre_lookback_count:,} -> {_post_lookback_count:,} rows "
        f"(lookback_periods={intent.lookback_periods}, "
        f"time_col={_primary_time_col!r})"
    )

_filter_diagnostics = []
_segment_ids = compute_sampling_universe(
    frames=loaded_frames,
    entity_columns=_segment_entity_cols,
    primary_entity_dataset=_primary,
    filters=SAMPLE_FILTER_COLUMNS or None,
    diagnostics=_filter_diagnostics,
)
_unfiltered_entities = _population.total_entities
_effective_total = len(_segment_ids)

# Reporting-only: when SAMPLE_FILTER_COLUMNS narrows the universe, the
# "target rate" shown by render_population_markdown / the CI table should
# reflect the population that will actually be sampled (e.g. customers),
# not the pre-filter mix (which can include structural `target=0` rows
# such as never-customer prospects). Sampling logic is unchanged —
# prepare_sample_frame below already applies _segment_ids, so this block
# only swaps the displayed target_rate.
_display_population = _population
if (
    _segment_ids is not None
    and TARGET_COLUMN
    and TARGET_COLUMN in _target_df.columns
    and _effective_total > 0
):
    from dataclasses import replace as _dc_replace
    _filter_cols = [c for c in (ENTITY_COLUMN, TARGET_COLUMN) if c in _target_df.columns]
    _filtered_target_df = _segment_ids.filter_frame(_target_df[_filter_cols], ENTITY_COLUMN)
    _filtered_entity_level = _filtered_target_df.drop_duplicates(subset=[ENTITY_COLUMN])
    _filtered_rate = float(_filtered_entity_level[TARGET_COLUMN].mean())
    _display_population = _dc_replace(_population, target_rate=_filtered_rate)


# Minimum pass-rate guardrail — catches the class of bug where the filter
# silently evaluates to "all true" (e.g. F.expr limitation with subqueries)
# leaving the universe equal to the unfiltered total; also catches
# implausibly restrictive filter mis-evaluations. Percentage-based so it
# adapts to engagement size rather than hardcoding an absolute count.
# Override MIN_FILTER_PASS_RATE in a prior config cell if your filter is
# legitimately more (or less) selective than 1%.
_min_pass_rate = globals().get("MIN_FILTER_PASS_RATE", 0.01)
if SAMPLE_FILTER_COLUMNS and _unfiltered_entities > 0:
    _pass_rate = _effective_total / _unfiltered_entities
    if _effective_total == _unfiltered_entities:
        raise ValueError(
            f"SAMPLE_FILTER_COLUMNS is set but filter did not narrow the universe "
            f"({_effective_total:,} == unfiltered total). Likely F.expr failed to "
            f"evaluate a subquery — check filter_diagnostics: {_filter_diagnostics}"
        )
    if _pass_rate < _min_pass_rate:
        raise ValueError(
            f"Filter pass rate {_pass_rate:.2%} is below MIN_FILTER_PASS_RATE "
            f"({_min_pass_rate:.0%}). Universe={_effective_total:,} vs "
            f"total={_unfiltered_entities:,}. Either relax SAMPLE_FILTER_COLUMNS, "
            f"set a lower MIN_FILTER_PASS_RATE, or confirm the narrow universe is "
            f"intended for this engagement."
        )

_candidate_sizes = candidate_sample_sizes(_effective_total)
if _candidate_sizes:
    _estimates = estimate_sampling_accuracy(
        _effective_total, _display_population.target_rate, _candidate_sizes, _display_population.n_cohorts,
    )
    display(Markdown(render_population_markdown(
        _display_population, _effective_total, _segment_ids is not None, _unfiltered_entities, _estimates,
    )))
else:
    display(Markdown("**Sampling:** No entities detected. Verify ENTITY_COLUMN."))

display(Markdown(
    f"**Sampling universe:** {_effective_total:,} / {_unfiltered_entities:,} primary entities "
    f"pass filters (anchored at `{_primary}`)"
))

# Reporting-only: side-by-side sanity check when a filter is applied and
# the filtered target rate materially differs from the unfiltered rate.
# This flags configurations like SPS where the raw `account` table mixes
# prospects (never-customers, structurally target=0) with real customers,
# so reporting churn against the full table would mask the true rate on
# the modelled cohort.
if (
    _segment_ids is not None
    and _display_population is not _population
    and abs(_display_population.target_rate - _population.target_rate) > 0.005
):
    display(Markdown(
        f"**Target rate note:** filtered universe (sampled cohort) = "
        f"**{_display_population.target_rate:.1%}** vs raw `{_primary}` "
        f"table = {_population.target_rate:.1%} across all "
        f"{_population.total_entities:,} entities. The filtered rate is "
        f"the one to use when communicating the target rate of the "
        f"modelled population — the unfiltered rate mixes in rows that "
        f"SAMPLE_FILTER_COLUMNS excludes from training/scoring."
    ))

if SAMPLE_ENTITY_COUNT is not None and ENTITY_COLUMN and _target_df is not None and _effective_total > 0:
    _entity_only = prepare_sample_frame(
        _target_df, ENTITY_COLUMN, TARGET_COLUMN, _population.time_column,
        SAMPLE_STRATIFY_COLUMNS, _segment_ids,
    )
    _holdout_frac = max(0.0, min(1.0, HOLDOUT_FRACTION)) if HOLDOUT_FRACTION else 0.0

    _all_sampled_ids = stratified_entity_sample(
        entity_df=_entity_only,
        n_entities=SAMPLE_ENTITY_COUNT,
        entity_col=ENTITY_COLUMN,
        target_col=TARGET_COLUMN if TARGET_COLUMN in _entity_only.columns else None,
        time_col=_population.time_column if _population.time_column in _entity_only.columns else None,
        extra_strat_cols=SAMPLE_STRATIFY_COLUMNS,
    )

    if _holdout_frac > 0 and len(_all_sampled_ids) > 1:
        _sampled_ids, _holdout_ids = stratified_holdout_split(
            entity_df=_entity_only,
            entity_ids=_all_sampled_ids,
            holdout_fraction=_holdout_frac,
            entity_col=ENTITY_COLUMN,
            target_col=TARGET_COLUMN if TARGET_COLUMN in _entity_only.columns else None,
            time_col=_population.time_column if _population.time_column in _entity_only.columns else None,
            extra_strat_cols=SAMPLE_STRATIFY_COLUMNS,
        )
    else:
        _sampled_ids, _holdout_ids = _all_sampled_ids, []

    save_sample_ids(
        _namespace, _sampled_ids, _holdout_ids,
        entity_df=_entity_only, entity_col=ENTITY_COLUMN,
        target_col=TARGET_COLUMN if TARGET_COLUMN in _entity_only.columns else None,
    )

    import json as _json_sampling
    _sampling_diag = {
        "primary_entity_dataset": _primary,
        "unfiltered_entity_count": _unfiltered_entities,
        "universe_size": _effective_total,
        "filter_pass_rate": (_effective_total / _unfiltered_entities) if _unfiltered_entities else None,
        "sample_entity_count_requested": SAMPLE_ENTITY_COUNT,
        "train_count": len(_sampled_ids),
        "holdout_count": len(_holdout_ids),
        "filters": SAMPLE_FILTER_COLUMNS or {},
        "filter_diagnostics": [
            {"dataset": d.dataset, "filter_expr": d.filter_expr,
             "input_entities": d.input_entities, "output_entities": d.output_entities}
            for d in _filter_diagnostics
        ],
    }
    _diag_path = _namespace.sample_entity_ids_path.parent / "sampling_diagnostics.json"
    _diag_path.write_text(_json_sampling.dumps(_sampling_diag, indent=2, default=str))

    display(Markdown(render_sample_result_markdown(
        len(_all_sampled_ids), _effective_total, len(_sampled_ids), len(_holdout_ids),
        _holdout_frac, _population.time_column is not None, SAMPLE_STRATIFY_COLUMNS,
    )))

    project_context.sample_entity_count = SAMPLE_ENTITY_COUNT
    project_context.sample_stratify_columns = SAMPLE_STRATIFY_COLUMNS or None
    project_context.sample_filters = SAMPLE_FILTER_COLUMNS or None
    project_context.holdout_fraction = _holdout_frac if _holdout_frac > 0 else None
    project_context.save(context_path)
else:
    display(Markdown("**Sampling:** OFF (full data). Set `SAMPLE_ENTITY_COUNT` above to enable."))

if SAMPLE_FILTER_COLUMNS:
    display(Markdown(render_segment_filter_markdown(SAMPLE_FILTER_COLUMNS, _filter_diagnostics)))

[//]: # (cr:doc name='0_14_field_availability_audit' id=bde47c58)
## 0.14 Field Availability Audit (Optional)

Point-in-time correctness prevents using future *records*, but a feature
can still leak the outcome if it is populated only because termination is
already administratively determined (e.g., `CANCELLATION_REASON` appears
only when a contract is cancelled).

This **optional, one-time** analysis compares field population rates
between terminated and active service units (contracts, subscriptions) to
surface fields that encode the outcome through administrative side-effects.

**When to enable:** Set `RUN_FIELD_AVAILABILITY_AUDIT = True` once during
initial exploration. Review the output, then leave disabled for subsequent
runs. Results persist as YAML files in the run namespace.

See [docs/field_availability_audit.md](../docs/field_availability_audit.md)
for full design rationale, output format, and interpretation guide.

In [ ]:
# @cr:config name='field_availability_audit_config' id=316ca20a

# Set to True once during initial exploration to run the audit.
# Results are persisted — disable after the first run.
RUN_FIELD_AVAILABILITY_AUDIT = False

# Service unit detection (leave None for auto-detect from loaded_frames).
# Override when auto-detection picks the wrong table or columns.
SERVICE_UNIT_DATASET = None           # dataset name, e.g. "contract"
SERVICE_UNIT_ID_COLUMN = None         # unit primary key, e.g. "CONTRACT_ID"
SERVICE_UNIT_ANCHOR_COLUMN = None     # termination date, e.g. "CONTRACT_END_DATE"
SERVICE_UNIT_STATUS_COLUMN = None     # status column, e.g. "CONTRACT_STATUS"
SERVICE_UNIT_TERMINATED_STATUSES = [] # values meaning terminated, e.g. ["Cancelled", "Terminated"]
SERVICE_UNIT_START_COLUMN = None      # start date (optional), e.g. "CONTRACT_START_DATE"

# Audit parameters
AUDIT_MIN_TERMINATED_UNITS = 50       # minimum terminated units to produce meaningful stats
AUDIT_SUSPICION_THRESHOLD = 0.5       # score above this -> investigate; +0.2 -> exclude
AUDIT_LEAD_BUCKETS_DAYS = [0, 30, 90] # lead/lag bucket boundaries in days

# Manual exclusions applied on top of audit-detected ones
AUDIT_ADDITIONAL_EXCLUSIONS = []

In [ ]:
# @cr:code name='run_field_availability_audit' id=d02bfe24
if RUN_FIELD_AVAILABILITY_AUDIT:
    from customer_retention.analysis.auto_explorer.field_availability_audit import (
        run_field_availability_audit,
    )

    _audit_result = run_field_availability_audit(
        context=project_context,
        loaded_frames=loaded_frames,
        namespace=_namespace,
        service_unit_dataset=SERVICE_UNIT_DATASET,
        service_unit_id_column=SERVICE_UNIT_ID_COLUMN,
        service_unit_anchor_column=SERVICE_UNIT_ANCHOR_COLUMN,
        service_unit_status_column=SERVICE_UNIT_STATUS_COLUMN,
        service_unit_terminated_statuses=SERVICE_UNIT_TERMINATED_STATUSES,
        service_unit_start_column=SERVICE_UNIT_START_COLUMN,
        min_terminated_units=AUDIT_MIN_TERMINATED_UNITS,
        suspicion_threshold=AUDIT_SUSPICION_THRESHOLD,
        lead_buckets_days=AUDIT_LEAD_BUCKETS_DAYS,
        additional_exclusions=AUDIT_ADDITIONAL_EXCLUSIONS,
    )
else:
    display(Markdown(
        "**Field Availability Audit:** disabled. "
        "Set `RUN_FIELD_AVAILABILITY_AUDIT = True` above to run."
    ))

[//]: # (cr:doc name='0_15_schema_report' id=e4de33ca)
## 0.15 Dataset Schema Report (Optional)

Generates a Markdown file listing every column name and type for each
registered dataset, matching the format of `docs/sps_table_descriptions.md`.

Use this to compare the actual schema of your data sources against
documented expectations and identify missing or mistyped columns.

In [ ]:
# @cr:config name='schema_report_config' id=37ab7bbb

# Set to True to generate a dataset schema report (column names + types)
# matching the format of docs/sps_table_descriptions.md.
# Useful for comparing actual schema against documented columns.
GENERATE_SCHEMA_REPORT = False

# Output filename for the generated report (written to experiments/docs/).
SCHEMA_REPORT_FILENAME = "actual_table_descriptions.md"

# Set to True to auto-generate column descriptions via a Databricks
# Foundation Model endpoint (only runs on Databricks; ignored locally).
GENERATE_LLM_DESCRIPTIONS = False

# Serving endpoint name for LLM description generation.
# None = auto-detect the best available chat endpoint on the workspace.
LLM_ENDPOINT = None

In [ ]:
# @cr:code name='run_schema_report' id=204eae96
if GENERATE_SCHEMA_REPORT or GENERATE_LLM_DESCRIPTIONS:
    from customer_retention.analysis.auto_explorer.schema_report import (
        build_schema_report,
    )

    _result = build_schema_report(
        loaded_frames, datasets,
        llm_descriptions=GENERATE_LLM_DESCRIPTIONS,
        llm_endpoint=LLM_ENDPOINT,
    )

    if GENERATE_SCHEMA_REPORT:
        _report_path = get_experiments_dir() / "docs" / SCHEMA_REPORT_FILENAME
        _report_path.parent.mkdir(parents=True, exist_ok=True)
        _report_path.write_text(_result.text, encoding="utf-8")
    else:
        _report_path = None

    for _w in _result.warnings:
        display(Markdown(_w))

    _llm_note = (
        f" | LLM endpoint: `{_result.llm_endpoint_used}`"
        if _result.llm_endpoint_used else ""
    )
    if _report_path is not None:
        display(Markdown(
            f"**Schema Report:** written to `{_report_path}` "
            f"({_result.column_count} columns across "
            f"{_result.dataset_count} datasets{_llm_note})"
        ))

    # Bootstrap the column_descriptions sidecar from the SAME LLM call that
    # produced the schema report — no extra round-trip. Cycle 013 P11 fails
    # when this sidecar is empty; piggy-backing here keeps both artifacts
    # in lock-step so the interpretation layer (compile_predicate_prose,
    # archetype enrichment) gets business names without a separate cell.
    _seed_rows = []
    if _result.col_descriptions and _namespace is not None:
        from customer_retention.stages.causal.column_descriptions_writer import (
            ColumnDescriptionRow,
        )
        from customer_retention.stages.causal.interpretation import (
            write_column_descriptions_sidecar,
        )

        for _ds_name, _ds_descs in _result.col_descriptions.items():
            for _col, _desc in _ds_descs.items():
                if not _desc:
                    continue
                _seed_rows.append(ColumnDescriptionRow(
                    table=_ds_name,
                    column_name=_col,
                    business_definition=_desc,
                    source="llm_proposed",
                ))

        if _seed_rows:
            _seed_path = write_column_descriptions_sidecar(_namespace, _seed_rows)
            display(Markdown(
                f"**Column descriptions sidecar:** seeded with {len(_seed_rows)} "
                f"LLM-generated rows -> `{_seed_path}` "
                f"(source: `{_result.llm_endpoint_used}`)"
            ))
        else:
            display(Markdown(
                "**Column descriptions sidecar:** LLM returned no usable "
                "descriptions; sidecar not updated."
            ))

    # ------------------------------------------------------------------
    # Synthetic source-token gap-fill — runs only when feature_meta exists
    # (post-gold). `parse_aggregation_feature_name` assigns synthetic tokens
    # like 'event' / 'event_gap' / 'lifecycle' / 'recency' to grouped
    # features; these tokens never appear in any raw table so the schema-
    # report bootstrap above can't describe them. We harvest the unique set
    # from the feature_meta sidecar, route them through the same LLM
    # endpoint as a synthetic table, and merge into the column_descriptions
    # sidecar. Without this step compile_predicate_prose falls back to the
    # raw feature_name in narration (Cycle 013 D1 surfaced the gap).
    if (
        GENERATE_LLM_DESCRIPTIONS
        and _result.llm_endpoint_used
        and _namespace is not None
    ):
        from customer_retention.stages.causal.column_descriptions_writer import (
            ColumnDescriptionRow,
        )
        from customer_retention.stages.causal.interpretation import (
            load_column_descriptions_sidecar,
            load_feature_meta_sidecar,
            write_column_descriptions_sidecar,
        )

        _fm = load_feature_meta_sidecar(_namespace) or {}
        _existing_descs = load_column_descriptions_sidecar(_namespace) or {}
        _covered_cols = set()
        for _key, _row in _existing_descs.items():
            _col_name = getattr(_row, "column_name", None) or _key
            _covered_cols.add(str(_col_name))

        _candidate_tokens = set()
        for _row in _fm.values():
            for _src in (getattr(_row, "source_columns", None) or []):
                if not _src or _src in _covered_cols:
                    continue
                _candidate_tokens.add(_src)

        if _candidate_tokens:
            from customer_retention.analysis.auto_explorer.column_describer import (
                generate_column_descriptions,
            )

            _synthetic_label = f"synthetic:{_namespace.run_id}"
            _gap_descs = generate_column_descriptions(
                table_fqn=_synthetic_label,
                columns=sorted(_candidate_tokens),
                endpoint=_result.llm_endpoint_used,
            )
            _gap_rows = [
                ColumnDescriptionRow(
                    table=_synthetic_label,
                    column_name=_tok,
                    business_definition=_desc,
                    source="llm_proposed",
                )
                for _tok, _desc in _gap_descs.items()
                if _desc
            ]
            if _gap_rows:
                _all_rows = list(_existing_descs.values()) + _seed_rows + _gap_rows
                _gap_path = write_column_descriptions_sidecar(_namespace, _all_rows)
                display(Markdown(
                    f"**Column descriptions gap-fill:** added {len(_gap_rows)} "
                    f"synthetic-token descriptions ({sorted(_candidate_tokens)[:5]}…) "
                    f"-> `{_gap_path}`"
                ))
            else:
                display(Markdown(
                    f"**Column descriptions gap-fill:** {len(_candidate_tokens)} "
                    f"synthetic tokens needed descriptions but the LLM returned "
                    f"none. Sidecar unchanged."
                ))
elif GENERATE_SCHEMA_REPORT is False and GENERATE_LLM_DESCRIPTIONS is False:
    display(Markdown(
        "**Schema Report:** disabled. "
        "Set `GENERATE_SCHEMA_REPORT = True` (markdown) or "
        "`GENERATE_LLM_DESCRIPTIONS = True` (sidecar bootstrap) above to run."
    ))


[//]: # (cr:doc name='user_code_assimilation_summary_intro' id=5e1a0002)
## User-code Assimilation Summary

Final check before the next notebook: every imperative `@cr.register` cell
and every declarative `registry.add_*` call should appear below with Lane 1
(registered) and Lane 2 (in-cell side effect applied) both green. Rows
marked ❌ indicate user_code that was registered but never executed its
in-notebook mutation, which silently breaks exploration↔production parity
downstream.

In [ ]:
# @cr:code name='user_code_assimilation_summary' id=5e1a0003
from customer_retention.runtime import cr

cr.print_user_code_summary(datasets, _namespace)

In [ ]:
# @cr:code name='release_stage_memory' id=628a1a47
from customer_retention.core.compat import release_stage_memory

release_stage_memory()